# 🧠 HybridID — S5: ResNet50 CNN Eğitimi
**Fake vs Real görüntü sınıflandırma modeli**

| Adım | İçerik |
|------|--------|
| ADIM 1 | GPU kontrolü + kütüphaneler |
| ADIM 2 | Google Drive mount + dataset hazırlık |
| ADIM 3 | Model mimarisi (ResNet50 Transfer Learning) |
| ADIM 4 | FAZ 1 — Head eğitimi |
| ADIM 5 | FAZ 2 — Fine-tuning |
| ADIM 6 | Test değerlendirmesi + metrikler |
| ADIM 7 | Sonuçları Drive'a kaydet |

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 1 — GPU Kontrolü & Kütüphaneler
# ═══════════════════════════════════════════════
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import os, json, time
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)

print('=' * 60)
print('  HybridID — S5 CNN Eğitimi')
print('=' * 60)
print(f'TensorFlow  : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU listesi : {gpus}')
if gpus:
    print(f'✅ GPU HAZIR → {gpus[0].name}')
else:
    print('⚠️  GPU bulunamadı — Runtime > Change runtime type > T4 GPU seç!')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 2 — Google Drive Mount + Dataset
# ═══════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# ⚙️ AYAR: Drive'daki zip dosyanın tam yolu
ZIP_PATH = '/content/drive/MyDrive/processed.zip'
EXTRACT_DIR = '/content/processed'
OUTPUT_DIR = '/content/drive/MyDrive/HybridID_models'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Zip dosyasını bul (farklı klasördeyse otomatik ara)
if not os.path.exists(ZIP_PATH):
    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-name', 'processed.zip', '-type', 'f'],
        capture_output=True, text=True
    )
    found = result.stdout.strip().split('\n')
    if found and found[0]:
        ZIP_PATH = found[0]
        print(f'✅ Zip bulundu: {ZIP_PATH}')
    else:
        print('❌ processed.zip bulunamadı! Drive klasörünü kontrol et.')

# Dataset'i çıkar
if not os.path.exists(EXTRACT_DIR):
    print(f'Zip açılıyor: {ZIP_PATH} ...')
    !unzip -q "{ZIP_PATH}" -d /content/
    print('✅ Dataset hazır!')
else:
    print('✅ Dataset zaten mevcut, tekrar açılmıyor.')

# Kontrol
for split in ['train', 'val', 'test']:
    for cls in ['fake', 'real']:
        p = os.path.join(EXTRACT_DIR, split, cls)
        if os.path.exists(p):
            n = len(os.listdir(p))
            print(f'  {split}/{cls}: {n} görüntü')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 3 — Konfigürasyon & Veri Yükleyiciler
# ═══════════════════════════════════════════════
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

CONFIG = {
    'dataset_dir'        : EXTRACT_DIR,
    'img_size'           : (224, 224),
    'batch_size'         : 64,          # GPU'da 64 rahat çalışır
    'class_names'        : ['fake', 'real'],
    'dropout_rate'       : 0.5,
    'l2_lambda'          : 1e-4,
    'dense_units'        : 256,
    'phase1_epochs'      : 10,
    'phase1_lr'          : 1e-3,
    'phase2_epochs'      : 30,
    'phase2_lr'          : 1e-5,
    'unfreeze_from_layer': 143,
    'patience'           : 8,
    'model_dir'          : OUTPUT_DIR,
    'model_name'         : 'resnet50_hybridid.h5',
}

# --- Data generators ---
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.85, 1.15],
    shear_range=5,
)
eval_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    os.path.join(CONFIG['dataset_dir'], 'train'),
    target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
    class_mode='binary', shuffle=True, seed=42
)
val_gen = eval_datagen.flow_from_directory(
    os.path.join(CONFIG['dataset_dir'], 'val'),
    target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
    class_mode='binary', shuffle=False
)
test_gen = eval_datagen.flow_from_directory(
    os.path.join(CONFIG['dataset_dir'], 'test'),
    target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
    class_mode='binary', shuffle=False
)

print(f'\nSınıf indeksleri : {train_gen.class_indices}')
print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 4 — Model Mimarisi (ResNet50)
# ═══════════════════════════════════════════════
base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(*CONFIG['img_size'], 3)
)
base.trainable = False  # Faz-1: dondurulmuş

inputs  = keras.Input(shape=(*CONFIG['img_size'], 3), name='input_image')
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D(name='gap')(x)
x       = layers.BatchNormalization(name='bn_head')(x)
x       = layers.Dense(
              CONFIG['dense_units'], activation='relu',
              kernel_regularizer=regularizers.l2(CONFIG['l2_lambda']),
              name='dense_head')(x)
x       = layers.Dropout(CONFIG['dropout_rate'], name='dropout_head')(x)
outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

model = keras.Model(inputs, outputs, name='HybridID_ResNet50')

total_p     = model.count_params()
trainable_p = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Toplam parametre      : {total_p:,}')
print(f'Eğitilebilir parametre: {trainable_p:,}')
print(f'Mimari: ResNet50 (ImageNet) → GAP → BN → Dense(256) → Dropout → Sigmoid')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 5 — FAZ 1: Head Eğitimi
# ═══════════════════════════════════════════════
model_path = os.path.join(CONFIG['model_dir'], CONFIG['model_name'])

callbacks_p1 = [
    ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=CONFIG['patience'], restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['phase1_lr']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

print('─' * 50)
print('  FAZ 1: Head katmanı eğitimi (ResNet50 dondurulmuş)')
print('─' * 50)

t0 = time.time()
h1 = model.fit(
    train_gen,
    epochs=CONFIG['phase1_epochs'],
    validation_data=val_gen,
    callbacks=callbacks_p1,
    verbose=1
)
print(f'\n✅ Faz-1 tamamlandı: {(time.time()-t0)/60:.1f} dakika')
print(f'   En iyi val_accuracy: {max(h1.history["val_accuracy"]):.4f}')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 6 — FAZ 2: Fine-Tuning
# ═══════════════════════════════════════════════
base.trainable = True
for layer in base.layers[:CONFIG['unfreeze_from_layer']]:
    layer.trainable = False

trainable_now = sum(1 for l in base.layers if l.trainable)
print(f'Açılan ResNet50 katman sayısı: {trainable_now}')

callbacks_p2 = [
    ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=CONFIG['patience'], restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['phase2_lr']),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

print('─' * 50)
print(f'  FAZ 2: Fine-tuning (katman {CONFIG["unfreeze_from_layer"]}+ açık)')
print('─' * 50)

t0 = time.time()
h2 = model.fit(
    train_gen,
    epochs=CONFIG['phase2_epochs'],
    validation_data=val_gen,
    callbacks=callbacks_p2,
    verbose=1
)
print(f'\n✅ Faz-2 tamamlandı: {(time.time()-t0)/60:.1f} dakika')
print(f'   En iyi val_accuracy: {max(h2.history["val_accuracy"]):.4f}')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 7 — Test Seti Değerlendirmesi
# ═══════════════════════════════════════════════
print('=' * 60)
print('  TEST SETİ DEĞERLENDİRMESİ')
print('=' * 60)

test_gen.reset()
y_prob = model.predict(test_gen, verbose=1)
y_pred = (y_prob > 0.5).astype(int).flatten()
y_true = test_gen.classes

report = classification_report(y_true, y_pred, target_names=CONFIG['class_names'], output_dict=True)
auc    = roc_auc_score(y_true, y_prob)

print('\n📊 Sınıflandırma Raporu:')
print(classification_report(y_true, y_pred, target_names=CONFIG['class_names']))
print(f'ROC-AUC Skoru : {auc:.4f}')
print(f'Test Accuracy : {report["accuracy"]:.4f}')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 8 — Confusion Matrix & Eğitim Eğrileri
# ═══════════════════════════════════════════════

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CONFIG['class_names'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('HybridID — Confusion Matrix (Test Seti)', fontsize=14, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(CONFIG['model_dir'], 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'✅ Confusion matrix kaydedildi → {cm_path}')

# --- Eğitim Eğrileri ---
def concat(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

acc      = concat(h1, h2, 'accuracy')
val_acc  = concat(h1, h2, 'val_accuracy')
loss     = concat(h1, h2, 'loss')
val_loss = concat(h1, h2, 'val_loss')
auc_hist = concat(h1, h2, 'auc')
val_auc  = concat(h1, h2, 'val_auc')
epochs   = range(1, len(acc) + 1)
p1_end   = len(h1.history['accuracy'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('HybridID — Eğitim Eğrileri (Faz1 + Faz2)', fontsize=15, fontweight='bold')

for ax, (tr, vl, title) in zip(axes, [
    (acc, val_acc, 'Accuracy'),
    (loss, val_loss, 'Loss'),
    (auc_hist, val_auc, 'ROC-AUC')
]):
    ax.plot(epochs, tr, 'b-o', markersize=3, label='Train')
    ax.plot(epochs, vl, 'r-o', markersize=3, label='Val')
    ax.axvline(p1_end + 0.5, color='gray', linestyle='--', alpha=0.7, label='Faz 1/2 sınırı')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(CONFIG['model_dir'], 'training_curves.png')
plt.savefig(curves_path, dpi=150)
plt.show()
print(f'✅ Eğitim eğrileri kaydedildi → {curves_path}')

In [ ]:
# ═══════════════════════════════════════════════
# ADIM 9 — Sonuçları Drive'a Kaydet
# ═══════════════════════════════════════════════
history_all = {
    'phase1': {k: [float(v) for v in vals] for k, vals in h1.history.items()},
    'phase2': {k: [float(v) for v in vals] for k, vals in h2.history.items()},
}

eval_report = {
    'test_accuracy'  : float(report['accuracy']),
    'roc_auc'        : float(auc),
    'fake_precision' : float(report['fake']['precision']),
    'fake_recall'    : float(report['fake']['recall']),
    'fake_f1'        : float(report['fake']['f1-score']),
    'real_precision' : float(report['real']['precision']),
    'real_recall'    : float(report['real']['recall']),
    'real_f1'        : float(report['real']['f1-score']),
    'total_epochs'   : len(acc),
    'phase1_epochs'  : p1_end,
    'phase2_epochs'  : len(h2.history['accuracy']),
    'model'          : 'ResNet50 Transfer Learning',
    'dataset'        : f'Train:{train_gen.samples} Val:{val_gen.samples} Test:{test_gen.samples}',
}

# JSON kayıt
report_path  = os.path.join(CONFIG['model_dir'], 'evaluation_report.json')
history_path = os.path.join(CONFIG['model_dir'], 'training_history.json')

with open(report_path,  'w') as f: json.dump(eval_report,  f, indent=2)
with open(history_path, 'w') as f: json.dump(history_all, f, indent=2)

print('\n' + '=' * 60)
print('  ✅ TÜM SONUÇLAR DRIVE\'A KAYDEDİLDİ')
print('=' * 60)
print(f'  📁 Konum: {CONFIG["model_dir"]}')
print(f'  🧠 Model           : resnet50_hybridid.h5')
print(f'  📊 Eval raporu     : evaluation_report.json')
print(f'  📈 Eğitim geçmişi  : training_history.json')
print(f'  🖼️  Confusion matrix: confusion_matrix.png')
print(f'  📉 Eğitim eğrileri : training_curves.png')
print('=' * 60)
print(f'\n  🎯 Test Accuracy : {eval_report["test_accuracy"]:.4f} ({eval_report["test_accuracy"]*100:.2f}%)')
print(f'  🎯 ROC-AUC       : {eval_report["roc_auc"]:.4f}')
print(f'  🎯 Fake F1-Score  : {eval_report["fake_f1"]:.4f}')
print(f'  🎯 Real F1-Score  : {eval_report["real_f1"]:.4f}')
print('=' * 60)